### Preprocessing
Defining the 6 moral labels and preparing the moral-vector. We prepare the moral-vector by parsing it. We do that by filtering out any moral-vector that's not a list or doesn't have length of 6.

In [6]:
import pandas as pd
import numpy as np
import ast

df = pd.read_csv('./dataset/MIC.csv')

# keep only rows with usable text and a non-null moral label
df = df.dropna(subset=['Q', 'A', 'moral', 'moral-vector']).copy()

df['QA_clean'] = df['Q'].astype(str).str.strip() + ' ' + df['A'].astype(str).str.strip()

def safe_parse_vector(s):
    try:
        v = ast.literal_eval(s)
        if isinstance(v, list) and len(v) == 6:
            return v
    except (ValueError, SyntaxError):
        pass
    return None

df['moral_vector_parsed'] = df['moral-vector'].apply(safe_parse_vector)

n_before = len(df)
df = df.dropna(subset=['moral_vector_parsed']).copy()
print(f"Dropped {n_before - len(df)} rows with malformed moral-vector")

MORAL_LABELS = ['care', 'fairness', 'liberty', 'loyalty', 'authority', 'sanctity']

print(df.shape)
print(df[['moral', 'moral_vector_parsed']].head())
print(df['split'].value_counts())


Dropped 3 rows with malformed moral-vector
(108192, 15)
                moral             moral_vector_parsed
0             loyalty  [0.0, 0.0, 0.0, 1.0, 0.0, 0.0]
1       care|fairness  [1.0, 1.0, 0.0, 0.0, 0.0, 0.0]
2                care  [1.0, 0.0, 0.0, 0.0, 0.0, 0.0]
3  authority|sanctity  [0.0, 0.0, 0.0, 0.0, 1.0, 1.0]
4           authority  [0.0, 0.0, 0.0, 0.0, 1.0, 0.0]
split
train    86528
dev      10842
test     10817
Name: count, dtype: int64


Cleaning and Tokenization

In [7]:
import re

def simple_preprocess(text: str) -> str:
    text = text.lower()
    text = re.sub(r"\s+", " ", text).strip()
    return text

TOKEN_PATTERN = re.compile(r"[a-z0-9]+(?:'[a-z]+)?|[.,!?;:]")

def tokenize(text: str):
    return TOKEN_PATTERN.findall(text)

df['QA_processed'] = df['QA_clean'].apply(simple_preprocess)

# sanity check
sample = df['QA_processed'].iloc[0]
print(sample)
print(tokenize(sample))


am i a bad bf, weird, or just going insane? i don't think you're a bad bf or weird or going insane. i think you just need to talk to him about it.
['am', 'i', 'a', 'bad', 'bf', ',', 'weird', ',', 'or', 'just', 'going', 'insane', '?', 'i', "don't", 'think', "you're", 'a', 'bad', 'bf', 'or', 'weird', 'or', 'going', 'insane', '.', 'i', 'think', 'you', 'just', 'need', 'to', 'talk', 'to', 'him', 'about', 'it', '.']


Train test split 

In [9]:
from sklearn.model_selection import train_test_split

# X = text, y = multi-label binary matrix (already built as moral_vector_parsed)
train_df = df[df['split'] == 'train'].copy()
test_df  = df[df['split'] == 'test'].copy()

X_train_text = train_df['QA_processed'].tolist()
X_test_text  = test_df['QA_processed'].tolist()

Y_train = np.array(train_df['moral_vector_parsed'].tolist())
Y_test  = np.array(test_df['moral_vector_parsed'].tolist())

print(f"Train: {len(X_train_text)} docs, Y_train shape: {Y_train.shape}")
print(f"Test:  {len(X_test_text)} docs,  Y_test shape:  {Y_test.shape}")

# label frequency in each split, sanity check against MORAL_LABELS order
print("\nLabel frequency (train):")
for i, label in enumerate(MORAL_LABELS):
    print(f"  {label:10s}: {int(Y_train[:, i].sum())}")

print("\nLabel frequency (test):")
for i, label in enumerate(MORAL_LABELS):
    print(f"  {label:10s}: {int(Y_test[:, i].sum())}")


Train: 86528 docs, Y_train shape: (86528, 6)
Test:  10817 docs,  Y_test shape:  (10817, 6)

Label frequency (train):
  care      : 46725
  fairness  : 18947
  liberty   : 17496
  loyalty   : 17296
  authority : 16170
  sanctity  : 10140

Label frequency (test):
  care      : 5823
  fairness  : 2357
  liberty   : 2218
  loyalty   : 2210
  authority : 1951
  sanctity  : 1235


### TF-IDF vectorization
Each sample row will have a vector, which is an array of weights, representing all the tokens in the vocabulary. These weights are different for each sample.

In [10]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(
    tokenizer=tokenize,
    preprocessor=None,   # already lowercased/cleaned in QA_processed
    token_pattern=None,  # silence sklearn's warning since we supply our own tokenizer
    lowercase=False,
)

X_train = vectorizer.fit_transform(X_train_text)
X_test  = vectorizer.transform(X_test_text)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape:  {X_test.shape}")
print(f"Vocabulary size: {len(vectorizer.vocabulary_)}")


X_train shape: (86528, 23079)
X_test shape:  (10817, 23079)
Vocabulary size: 23079


Naive Bayes

In [11]:
from sklearn.multiclass import OneVsRestClassifier
from sklearn.naive_bayes import MultinomialNB
import time

nb_model = OneVsRestClassifier(MultinomialNB())

start = time.time()
nb_model.fit(X_train, Y_train)
print(f"Trained in {time.time() - start:.1f}s")

Y_pred_nb = nb_model.predict(X_test)
Y_proba_nb = nb_model.predict_proba(X_test)

print(f"Y_pred_nb shape: {Y_pred_nb.shape}")
print("\nSample prediction vs actual (first test doc):")
print("Text:", X_test_text[0][:150])
print("Predicted probs:", dict(zip(MORAL_LABELS, Y_proba_nb[0].round(3))))
print("Predicted labels:", [MORAL_LABELS[i] for i, v in enumerate(Y_pred_nb[0]) if v == 1])
print("Actual labels:   ", [MORAL_LABELS[i] for i, v in enumerate(Y_test[0]) if v == 1])


Trained in 0.2s
Y_pred_nb shape: (10817, 6)

Sample prediction vs actual (first test doc):
Text: would you rather have your child be less attractive but extremely intelligent or extremely attractive but less intelligent? i'd rather have my child b
Predicted probs: {'care': np.float64(0.552), 'fairness': np.float64(0.237), 'liberty': np.float64(0.152), 'loyalty': np.float64(0.099), 'authority': np.float64(0.07), 'sanctity': np.float64(0.113)}
Predicted labels: ['care']
Actual labels:    ['care', 'fairness', 'loyalty', 'authority']


Logistic Regression

In [12]:
from sklearn.linear_model import LogisticRegression

lr_model = OneVsRestClassifier(LogisticRegression(max_iter=1000))

start = time.time()
lr_model.fit(X_train, Y_train)
print(f"Trained in {time.time() - start:.1f}s")

Y_pred_lr = lr_model.predict(X_test)
Y_proba_lr = lr_model.predict_proba(X_test)

print(f"Y_pred_lr shape: {Y_pred_lr.shape}")
print("\nSame sample prediction:")
print("Predicted probs:", dict(zip(MORAL_LABELS, Y_proba_lr[0].round(3))))
print("Predicted labels:", [MORAL_LABELS[i] for i, v in enumerate(Y_pred_lr[0]) if v == 1])
print("Actual labels:   ", [MORAL_LABELS[i] for i, v in enumerate(Y_test[0]) if v == 1])


Trained in 5.9s
Y_pred_lr shape: (10817, 6)

Same sample prediction:
Predicted probs: {'care': np.float64(0.652), 'fairness': np.float64(0.388), 'liberty': np.float64(0.149), 'loyalty': np.float64(0.11), 'authority': np.float64(0.086), 'sanctity': np.float64(0.224)}
Predicted labels: ['care']
Actual labels:    ['care', 'fairness', 'loyalty', 'authority']


In [13]:
from sklearn.metrics import (
    classification_report,
    accuracy_score,
    hamming_loss,
    precision_recall_fscore_support,
)

def evaluate(name, Y_true, Y_pred):
    print(f"{'='*20} {name} {'='*20}")
    print(classification_report(Y_true, Y_pred, target_names=MORAL_LABELS, zero_division=0))

    subset_acc = accuracy_score(Y_true, Y_pred)          # exact match across all 6 labels
    hl = hamming_loss(Y_true, Y_pred)                     # fraction of wrong label decisions
    micro = precision_recall_fscore_support(Y_true, Y_pred, average='micro', zero_division=0)
    macro = precision_recall_fscore_support(Y_true, Y_pred, average='macro', zero_division=0)

    print(f"Subset accuracy (exact match): {subset_acc:.4f}")
    print(f"Hamming accuracy (1 - Hamming loss): {1 - hl:.4f}")
    print(f"Micro  P/R/F1: {micro[0]:.4f} / {micro[1]:.4f} / {micro[2]:.4f}")
    print(f"Macro  P/R/F1: {macro[0]:.4f} / {macro[1]:.4f} / {macro[2]:.4f}")
    print()
    return {
        'subset_accuracy': subset_acc,
        'hamming_accuracy': 1 - hl,
        'micro_precision': micro[0], 'micro_recall': micro[1], 'micro_f1': micro[2],
        'macro_precision': macro[0], 'macro_recall': macro[1], 'macro_f1': macro[2],
    }

nb_metrics = evaluate("Naive Bayes", Y_test, Y_pred_nb)
lr_metrics = evaluate("Logistic Regression", Y_test, Y_pred_lr)

comparison = pd.DataFrame([nb_metrics, lr_metrics], index=['Naive Bayes', 'Logistic Regression'])
print(comparison.round(4))


==================== Naive Bayes ====================
              precision    recall  f1-score   support

        care       0.63      0.77      0.69      5823
    fairness       0.52      0.03      0.05      2357
     liberty       0.42      0.02      0.04      2218
     loyalty       0.71      0.06      0.11      2210
   authority       0.56      0.07      0.12      1951
    sanctity       0.33      0.00      0.00      1235

   micro avg       0.62      0.31      0.41     15794
   macro avg       0.53      0.16      0.17     15794
weighted avg       0.56      0.31      0.30     15794
 samples avg       0.44      0.34      0.37     15794

Subset accuracy (exact match): 0.2391
Hamming accuracy (1 - Hamming loss): 0.7859
Micro  P/R/F1: 0.6207 / 0.3089 / 0.4125
Macro  P/R/F1: 0.5279 / 0.1588 / 0.1706

==================== Logistic Regression ====================
              precision    recall  f1-score   support

        care       0.65      0.70      0.67      5823
    fairness   